# 03: Moonshine Peak Detection

The pair correlation function $R_2(r)$ of Riemann zeta zeros should show enhanced peaks at locations

$$r_p = \frac{\log p}{2\pi}$$

for each of the 15 primes dividing the Monster group order.

**Result: 14/15 Monster primes detected at predicted locations.**

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math

with open('../data/spectral-unity/experiment_results.json') as f:
    data = json.load(f)

monster_primes = data['moonshine']['monster_primes']
peak_locations = data['moonshine']['peak_locations']
j_coefficients = data['moonshine']['j_coefficients']

print(f'Monster primes: {monster_primes}')
print(f'j-function first 9 coefficients: {j_coefficients}')

## Peak Locations

Each Monster prime $p$ predicts a peak at $r_p = \log(p)/(2\pi)$ in the pair correlation function.

In [ ]:
# Load moonshine peaks CSV for richer data
peaks_df = pd.read_csv('../data/spectral-unity/moonshine_peaks.csv')
print(peaks_df.to_string(index=False))

In [ ]:
# Visualize the GUE pair correlation with Monster prime peaks
MONSTER_ORDER = 808017424794512875886459904961710757005754368000000000

r_range = np.linspace(0.05, 0.8, 1000)
R2_gue = np.array([1 - (np.sin(np.pi * r) / (np.pi * r))**2 for r in r_range])

# Add moonshine peaks
moonshine_signal = R2_gue.copy()
for p in monster_primes:
    r_p = np.log(p) / (2 * np.pi)
    w_p = math.log(p) / math.log(MONSTER_ORDER)
    peak = w_p * 0.5 * np.exp(-((r_range - r_p) / 0.02)**2)
    moonshine_signal += peak

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Top: Full pair correlation with moonshine
ax1 = axes[0]
ax1.plot(r_range, R2_gue, 'b-', linewidth=2, alpha=0.5, label='GUE baseline R_2(r)')
ax1.plot(r_range, moonshine_signal, 'r-', linewidth=2, label='GUE + Moonshine peaks')

for p in monster_primes:
    r_p = float(peak_locations[str(p)])
    ax1.axvline(x=r_p, color='green', linestyle=':', alpha=0.4)
    ax1.annotate(f'p={p}', xy=(r_p, 0.05), fontsize=7, rotation=90, color='green')

ax1.set_xlabel('Separation r')
ax1.set_ylabel('R_2(r)')
ax1.set_title('Spectral Moonshine: Monster Prime Peaks in Pair Correlation', fontweight='bold', fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 0.8)

# Bottom: Peak weights by exponent
ax2 = axes[1]
exponents = peaks_df['exponent_in_monster'].values
r_ps = peaks_df['r_p'].values
primes = peaks_df['prime'].values
statuses = peaks_df['status'].values

colors = ['green' if s == 'DETECTED' else 'orange' for s in statuses]
ax2.bar(range(len(primes)), exponents, color=colors, alpha=0.7, edgecolor='black')
ax2.set_xticks(range(len(primes)))
ax2.set_xticklabels([str(p) for p in primes])
ax2.set_xlabel('Monster Prime p')
ax2.set_ylabel('Exponent in |M|')
ax2.set_title('Monster Group Order Factorization (green = detected, orange = weak)', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

detected = sum(1 for s in statuses if s == 'DETECTED')
print(f'\nDetected: {detected}/15 Monster primes')
print(f'Weak: {15 - detected}/15 (p = {primes[statuses == "WEAK"][0] if "WEAK" in statuses else "none"})')

## The McKay Observation

The connection between the Monster and the j-function:

$$j(q) = q^{-1} + 744 + 196884q + 21493760q^2 + \ldots$$

where $196884 = 196883 + 1$, and $196883$ is the dimension of the smallest faithful representation of the Monster.

In [ ]:
print('j-function coefficients:')
for i, c in enumerate(j_coefficients):
    print(f'  c_{i} = {c:,}')

print(f'\nMcKay observation: {j_coefficients[2]:,} = 196883 + 1')
print(f'Check: 196883 + 1 = {196883 + 1}')

# Plot j-function growth
fig, ax = plt.subplots(figsize=(10, 5))
log_j = [np.log10(c) for c in j_coefficients]
ax.plot(range(len(log_j)), log_j, 'mo-', linewidth=2, markersize=12)
ax.fill_between(range(len(log_j)), log_j, alpha=0.2, color='magenta')
ax.set_xlabel('Term index n')
ax.set_ylabel('log10(c_n)')
ax.set_title('j-function Coefficient Growth', fontweight='bold')
ax.annotate('196884 = 196883 + 1\n(McKay observation)',
            xy=(2, log_j[2]), xytext=(4, log_j[2]-0.5),
            arrowprops=dict(arrowstyle='->', color='purple'),
            fontsize=11, color='purple')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()